In [1]:
import os, sys 
from pathlib import Path
from os.path import dirname, realpath
script_dir = Path(dirname(realpath('.')))
module_dir = str(script_dir)
sys.path.insert(0, module_dir + '/modules')
import numpy as np
import utility as ut
import matplotlib.pyplot as plt
import pandas as pd
import warnings
from scipy import stats
import seaborn as sns
import sample as sm
import chain as arch 
import torch
# warnings.filterwarnings('ignore')

In [2]:
Uo = np.load('../data/L63-trajectories/train.npy')[:, :20000]
Vo = np.load('../data/L63-trajectories/test.npy')
L0, L1 = 0.4, 3.5
D, D_r, B = 3, 256, 2
beta = 4e-5
m = 500

In [3]:
B = 1
rf1 = arch.DeepRF(D_r, B, L0, L1, Uo, beta, name='test', save_folder='../data')
rf1.init()
tau1 = rf1.compute_tau_f_(Vo[0:m])

AttributeError: 'Chain' object has no attribute ''

In [ ]:
B = 2
rf2 = arch.DeepRF(D_r, B, L0, L1, Uo, beta, name='test', save_folder='../data')
rf2.init()
tau2 = rf2.compute_tau_f_(Vo[0:m])

In [5]:
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

B = 3
rf3 = arch.DeepRF(D_r, B, L0, L1, Uo, beta, name='test', save_folder='../data')
rf3.init()
tau3 = rf3.compute_tau_f_(Vo[0:m])

Time taken by sample is 0.0062 seconds
Time taken by sample is 0.0105 seconds
Time taken by sample is 0.0105 seconds
Time taken by compute_tau_f_ is 25.1681 seconds


In [6]:
B = 10
rf10 = arch.DeepRF(D_r, B, L0, L1, Uo, beta, name='test', save_folder='../data')
rf10.init()
tau10 = rf10.compute_tau_f_(Vo[0:m])

Time taken by sample is 0.0062 seconds
Time taken by sample is 0.0073 seconds
Time taken by sample is 0.0048 seconds
Time taken by sample is 0.0048 seconds
Time taken by sample is 0.0119 seconds
Time taken by sample is 0.0047 seconds
Time taken by sample is 0.0104 seconds
Time taken by sample is 0.0049 seconds
Time taken by sample is 0.0107 seconds
Time taken by sample is 0.0046 seconds
Time taken by compute_tau_f_ is 79.8860 seconds


In [1]:
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)
sns.histplot(tau1[1], ax=ax, label=r'depth=1, $\mathbb{E}[\tau_f]$'+f'={tau1[1].mean():.2f}', alpha=0.7, stat='probability')
sns.histplot(tau2[1], ax=ax, label=r'depth=2, $\mathbb{E}[\tau_f]$'+f'={tau2[1].mean():.2f}', alpha=0.7, stat='probability')
sns.histplot(tau3[1], ax=ax, label=r'depth=3, $\mathbb{E}[\tau_f]$'+f'={tau3[1].mean():.2f}', alpha=0.7, stat='probability')
sns.histplot(tau10[1], ax=ax, label=r'depth=10, $\mathbb{E}[\tau_f]$'+f'={tau10[1].mean():.2f}', alpha=0.7, stat='probability')
ax.legend()
ax.set_title(fr'$\tau_f$, architecture=chain, $D_r$={D_r}')
plt.savefig(f'../data/plots/chain-tau_f-D_r-{D_r}.png', bbox_inches='tight', dpi=300)

NameError: name 'plt' is not defined

In [16]:
rf3.net.inner[1].register_forward_hook(get_activation('inner.1'))
x = torch.from_numpy(Uo[:, 0])
output = rf3.net(x)
0.4<torch.abs(activation['inner.1']) 

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, Tr

In [9]:
Uo[:, 0].shape

(3,)

In [19]:
for name, m in rf3.net.named_modules():
    print(name)


inner
inner.0
inner.1
inner.2
outer
outer.0
outer.1
outer.2


In [20]:
rf3.net.inner.register_forward_hook(get_activation('inner'))